# R01 · Why does a scan need a processing workflow?

**Reading seminar · Weeks 1–2.** Open the assigned papers alongside this notebook. Record your responses in a personal copy and bring the evidence sheet to discussion. AI can help explain unfamiliar terms; your reading of the source guides the interpretation.

**Method:** [three-pass reading guide](../../curriculum/papers/READING_METHOD.md). **Worksheet:** [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). **Assignment:** [A1 figure brief](../../curriculum/coursework/PAPER_TO_EXPERIMENT.md#a1--first-pass-figure-brief).

---

### Seminar Framing: From Raw Scanner Voxels to Anatomically Meaningful Coordinates

When an MRI scanner finishes an acquisition, it does not output a clean map of "brain function" or "cortical thickness." It outputs a 3D or 4D discrete lattice of numbers stored in an array $(i, j, k, t)$ accompanied by a header affine matrix $\mathbf{A}$ and acquisition metadata. Between that raw array and any downstream statistical map or deep-learning model stands a series of **spatial and temporal transformations**: skull stripping, bias-field intensity correction, slice-timing interpolation, head-motion realignment, susceptibility distortion correction ($B_0$ unwarping), cross-modal coregistration, nonlinear spatial normalization to a template, and resampling onto a target grid.

Every single transformation alters image intensities, coordinate mappings, spatial smoothness, or noise covariance. In this seminar, we pair a foundational **end-to-end preprocessing workflow** (`PP01`: *fMRIPrep*) with a recent **foundation-scale learned registration component** (`PP04`: *BrainMorph*), and situate both within the broader Processing strand ([SynthSeg `PP03`](../../curriculum/papers/processing.md#pp03), [FastSurfer `PP07`](../../curriculum/papers/processing.md#pp07), and [QSIPrep `PP08`](../../curriculum/papers/processing.md#pp08)).


## Read this pair first: Glass-Box Workflow vs. Learned Registration Component

- **Essential Foundational Paper (`PP01`):** Esteban et al., *fMRIPrep: a robust preprocessing pipeline for functional MRI* (Nature Methods, 2019; 16:111–116). [Full paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC6319393/) · [PP01 coursework card](../../curriculum/papers/processing.md#pp01).
- **Recent Frontier Method (`PP04`):** Wang et al., *BrainMorph: A Foundational Keypoint Model for Robust and Flexible Brain MRI Registration* (MELBA, 2025; first preprint 2024). [Versioned paper record](https://arxiv.org/abs/2405.14019v3) · [PP04 coursework card](../../curriculum/papers/processing.md#pp04).

**First-pass reading targets:**
1. **In `PP01` (fMRIPrep):** Abstract, Introduction, Results sections on the *adaptive workflow* and *visual reports*, **Figure 1** (workflow graph and visual QC checkpoints), **Figure 3** (spatial smoothness and spatial normalization consistency on the `DS000030` stop-signal dataset compared with FSL `feat`, alongside **Figure 4** on first-level and group statistical activation maps), and Methods on *manual quality assessment* and *one-shot concatenated interpolation*.
2. **In `PP04` (BrainMorph):** Abstract, Introduction, Section 3 (keypoint detector architecture and closed-form/TPS transform solver), **Figure 1** (end-to-end keypoint extraction $\to$ transformation estimation $\to$ grid resampling overview), **Figures 2–5 & Section 5** (datasets, ROI Dice overlap, Hausdorff distance, and runtime across synthetic initial rotation angles against ITK-Elastix, ANTs, SynthMorph, and EasyReg), and Section 7 (Discussion and limitations of 3D Thin-Plate Splines for aligning fine cortical geometry at low initial misalignment on skull-stripped data).

### Side-by-Side Architectural Comparison (`PP01` vs. `PP04` & Processing Strand Context)

| Analytical Dimension | `PP01` · *fMRIPrep* (Esteban et al., 2019) | `PP04` · *BrainMorph* (Wang et al., 2025) | Connections to `PP03`, `PP07`, `PP08` |
|---|---|---|---|
| **Primary System Role** | **Complete, adaptive BIDS preprocessing workflow** for structural ($T_1\text{w}/T_2\text{w}$) and functional (BOLD EPI) MRI prior to statistical modeling. | **Specialized spatial registration component** (pairwise 3D structural/multimodal MRI alignment via learned 3D keypoints). | [`PP08` (QSIPrep)](../../curriculum/papers/processing.md#pp08) extends `PP01`'s adaptive BIDS/one-shot philosophy to diffusion MRI q-space and b-vector rotation. |
| **Input Contract & Metadata Required** | BIDS-validated dataset: 3D anatomical + 4D functional NIfTI volumes **plus BIDS JSON sidecars** (`RepetitionTime` required; `PhaseEncodingDirection`, `EffectiveEchoSpacing`, `SliceTiming`, and fieldmaps conditionally drive adaptive correction steps when present). | Pair of 3D volumetric MRI scans (moving image $\mathbf{I}_m$ and fixed image $\mathbf{I}_f$) resampled/cropped to a standardized $1\text{ mm}^3$ conform grid. | [`PP03` (SynthSeg)](../../curriculum/papers/processing.md#pp03) & [`PP07` (FastSurfer)](../../curriculum/papers/processing.md#pp07) segment 3D structural scans; `PP04` uses SynthSeg parcellations during training supervision. |
| **Core Mathematical / Algorithmic Mechanism** | Orchestrates best-in-class tools (ANTs `antsRegistration`/`antsBrainExtraction`, FreeSurfer `recon-all`, FSL `mcflirt`/`topup`, Nipype DAGs) and **concatenates all spatial transforms** ($\mathbf{T}_{\text{motion}} \circ \mathbf{T}_{\text{SDC}} \circ \mathbf{T}_{\text{coreg}} \circ \mathbf{T}_{\text{MNI}}$) into a **single Lanczos interpolation step**. | Truncated 3D U-Net backbone trained on $>100,000$ 3D volumes ($50,183$ unique training scans doubled with and without skull stripping across $14,274$ subjects from $25$ training datasets) extracts $K \in \{128, 256, 512\}$ translation-equivariant 3D keypoints via center-of-mass layers; solves closed-form **Rigid/Affine** or regularized **Thin-Plate Spline (TPS)** transforms from point correspondences. | `PP07` shows why volumetric registration/segmentation Dice alone is insufficient for cortical morphometry—requiring spherical topological genus-0 mesh reconstruction. |
| **Primary Output Artifacts** | Preprocessed volumetric and surface-sampled BOLD derivatives in native, $T_1\text{w}$, and template spaces, a TSV table of time-series confounds (FD, DVARS, tCompCor, aCompCor, global signals), and **self-contained HTML visual QC reports** with dynamic and static mosaic views. | Estimated spatial coordinate mapping $\phi: \Omega_f \to \Omega_m$, warped moving image $\mathbf{I}_m \circ \phi$, and interpretable 3D anatomical keypoint coordinates. | `PP08` outputs preprocessed dMRI volumes, rotated b-tables, and interactive dMRI QC reports. |
| **How Success Is Evaluated in the Source** | Two-phase expert visual quality ratings on a $0\text{--}3$ scale across $7$ checkpoints (Phase I: $30$ MRIQC-flagged low-quality datasets; Phase II: $54$ OpenfMRI datasets, $325$ participants total), plus `DS000030` ($N = 257$) comparisons against FSL `feat` on spatial smoothness (Fig. 3A), normalization variance (Fig. 3B), and task-GLM maps (Fig. 4). | Anatomical ROI **Dice overlap** ($\uparrow$), **Hausdorff surface distance (HD)** ($\downarrow$), and GPU/CPU inference runtime across synthetic rotation angles on normal (`IXI`), longitudinal (`OASIS2`), and pathological (`BraTS` glioma, `MS-60` lesion) cohorts. | `PP03` evaluates robustness across arbitrary clinical contrasts and thick-slice resolutions without retraining. |
| **Explicit Limitations & Failure Modes** | Analysis-agnostic by design (does **not** perform smoothing or nuisance denoising decisions for the user); paper explicitly notes limitations on narrow-FOV acquisitions, infant/pediatric brains, and brains with large focal lesions/resections, plus reliance on visual QC inspection. | On skull-stripped scans with small initial misalignment, 3D Thin-Plate Spline warps from clustered keypoints underperform dense deformable baselines (**ANTs SyN** and **SynthMorph**) on fine cortical geometry; scaling $K$ from $128$ to $512$ yields minimal change, so the authors note BrainMorph can serve as a fast coarse initializer before ANTs. | Every automated tool (`PP01`–`PP08`) can fail silently on out-of-distribution pathology or corrupted headers without human visual QC. |


## Forensic Figure Walkthrough & Path-Drawing Exercise

Before filling in your comparison table, inspect the assigned figures in both papers with a forensic eye:

### 1. How to Read `PP01` (fMRIPrep) Figure 1 and Figure 3
- **Figure 1 (Adaptive Workflow & Visual Reports):** Trace the two parallel tracks—*Anatomical Workflow* (brain extraction, tissue segmentation, surface reconstruction, spatial normalization to MNI) and *Functional Workflow* (reference volume estimation, head-motion estimation, slice-time correction, susceptibility distortion correction, coregistration to $T_1\text{w}$).
  - *Crucial observation:* Notice that estimating a transformation (computing a $4 \times 4$ matrix or deformation field) is separated from **resampling** (interpolating voxel intensities onto a new grid). Why does fMRIPrep compose all spatial transforms and resample the BOLD series **only once**? Because every sequential interpolation step acts as a spatial low-pass filter that artificially blurs the image (fMRIPrep applies a single Lanczos interpolation across the composed transform).
- **Figure 3 (Smoothness & Spatial Normalization Consistency on `DS000030`):** Look at the comparison between `fMRIPrep` and `FSL feat` on the `DS000030` stop-signal dataset ($N = 257$). Check the axes and units: spatial smoothness in Fig. 3A is measured in physical millimeters ($\text{FWHM}$), showing that `FSL feat` outputs are intrinsically smoother before explicit smoothing is applied, while Fig. 3B shows lower across-participant anatomical boundary variance for `fMRIPrep`.

### 2. How to Read `PP04` (BrainMorph) Figure 1 and Figures 2–5
- **Figure 1 (Keypoint Registration Architecture):** Trace the three distinct mathematical objects:
  1. **Feature / Keypoint Extraction:** A 3D convolutional network maps input volume $\mathbf{I} \in \mathbb{R}^{H \times W \times D}$ to $K$ non-negative ReLU activation maps, and a spatial center-of-mass layer computes continuous 3D keypoint coordinates $\mathbf{P} \in \mathbb{R}^{K \times 3}$.
  2. **Transform Estimation (Solver):** Given moving keypoints $\mathbf{P}_m$ and fixed keypoints $\mathbf{P}_f$, a mathematical solver computes the transformation parameters $\phi_{\theta}$ in closed form (weighted least-squares for 12-DOF affine, or regularized Thin-Plate Splines for nonlinear warps).
  3. **Grid Resampling:** The spatial transformer grid sampler evaluates $\mathbf{I}_m$ at the transformed coordinates $\phi_{\theta}(\mathbf{x}_f)$ using trilinear interpolation (for continuous intensities) or nearest-neighbor interpolation (for discrete segmentation labels).
- **Figures 2–5 (Dice & Hausdorff Distance Across Initial Rotation Angles):** Watch the $x$-axis (synthetic rotation angle in degrees) and legend carefully. When does BrainMorph outperform dense deformable baselines (like ANTs SyN, SynthMorph, or EasyReg), and when does it trail them? Across large initial rotations, unstripped skulls, or brain tumors/lesions, keypoint matching remains robust. Conversely, at low initial misalignment on skull-stripped scans requiring fine cortical folding alignment, dense deformable fields (ANTs SyN and SynthMorph) outperform 3D Thin-Plate Splines.

---

## Draw the path of one scan (20-Minute Diagram Exercise)

Place a box for the **raw scanner image + BIDS sidecar** on the far left and a **proposed downstream statistical or clinical result** on the far right. Between them, add **only operations you can explicitly locate in the assigned source passages**.
- Mark every arrow with **what mathematical property changes**:
  - `[V]` = Voxel intensity values modified (e.g., bias correction, slice timing, interpolation);
  - `[C]` = Coordinate mapping / affine / warp estimated ($\mathbf{T}$ or $\phi$ computed without changing source pixels yet);
  - `[G]` = Sampling grid / resolution / matrix shape changed;
  - `[M]` = Metadata / confound TSV table generated;
  - `[QC]` = Human visual quality-control checkpoint.
- If you cannot tell from the first pass whether a step estimates a transform or resamples the grid, place an explicit **`[?]`** on that arrow rather than letting AI invent the detail.
- Finally, draw a dashed arrow showing where a learned registration component like `BrainMorph` (`PP04`) *could* plug into a broader workflow like `fMRIPrep` (`PP01`)—and label that arrow **"My proposed architectural integration (not jointly tested by the two papers)"**.


## Compare the evidence (Student Worksheet Table)

Complete all eight rows in your personal copy, citing the exact section, figure panel, or caption in each paper:

| Forensic Question | `PP01` · *fMRIPrep* (2019): Source locator & answer | `PP04` · *BrainMorph* (2025): Source locator & answer |
|---|---|---|
| **1. Input Contract:** What exact image arrays and non-image metadata are required? | | |
| **2. Proposed Operation:** Is this an end-to-end workflow orchestrator or a single learned component? What does it estimate vs. resample? | | |
| **3. Output & Provenance:** What files, transforms, or QC artifacts are produced? | | |
| **4. Quantitative Evaluation:** What is the observation unit ($N$), metric, and metric direction ($\uparrow$/$\downarrow$)? | | |
| **5. Baselines Compared:** Which existing software pipelines or neural networks are compared, and under what conditions? | | |
| **6. Difficult / Unestablished Cases:** Where does the method struggle, require fallback, or underperform a comparator? | | |
| **7. Downstream Impact:** How do choices here alter a later GLM (`P02`) or morphometry study (`P01`)? | | |
| **8. Human Responsibility:** Which scientific or QC decision still requires a human researcher after the code finishes with exit code `0`? | | |

### 100-Word Figure Brief per Paper
Choose one specific figure panel from `PP01` (e.g., Fig. 3 smoothness/activation panel) and one from `PP04` (e.g., Fig. 2 or Fig. 4 registration benchmark panel). Describe each in ~100 words. Explicitly state the **observation unit**, the **baseline comparator**, and **why an attractive visual overlay on a single brain slice is not a quantitative guarantee for an entire cohort**.


## Original Assignment: Accept or Investigate? (The "Black-Box AI Alignment" Audit)

Consider this realistic supervision scenario:

> **The Scenario:** You ask an AI coding assistant to align a participant's 3D brain scan and their binary hippocampus lesion mask to the MNI152 template so you can compare lesion volumes across 40 patients. Seconds later, the assistant returns a Python script that runs without error and displays a clean-looking 2D axial slice where the brain outline sits neatly inside the template box. However, the assistant **cannot tell you**:
> 1. Which image was treated as the *moving* source and which as the *fixed* reference (or whether the affine matrix $\mathbf{A}$ maps voxel-to-world or world-to-voxel);
> 2. Which exact MNI152 template version and resolution grid (`1 mm` vs. `2 mm`, linear vs. nonlinear asymmetric `ICBM152 2009c`) was used;
> 3. Whether the transformation was rigid (6 DOF), affine (12 DOF), or nonlinear deformable—and how the determinant of the Jacobian $|J_\phi|$ behaved;
> 4. Whether the binary lesion mask was resampled using **trilinear/continuous interpolation** or **nearest-neighbor interpolation** (`P01`).

### Case Study: How Passive Supervision Fails Here
If you accept the AI's output based on the single 2D screenshot:
- If the assistant used a **12-DOF affine or nonlinear warp** to match the template and you sum voxel counts in MNI space *without* multiplying by the Jacobian determinant $|J_\phi|$, your "lesion volumes" have been rescaled by global head-size normalization—erasing true atrophy differences!
- If the assistant resampled the binary `{0, 1}` mask with default **linear interpolation** and saved it as an integer array, boundary voxels with interpolated values like `0.42` were silently truncated to `0`, systematically shrinking every lesion's perimeter (`P01`)!
- If the assistant inverted the transform direction or flipped Left/Right (`RAS+` vs. `LPS+`), a left-hemisphere lesion is now analyzed in the right hemisphere!

**Your Task:** Write a 150–200 word audit response demanding the **minimum verifiable artifacts** required before you will accept the aligned scan and mask. Order your requests by **logical priority** (which ambiguity makes it impossible to interpret any subsequent number?). Connect each artifact request to a specific processing concept you will study in the course.

**Deliverables for `R01` ([A1 Figure Brief](../../curriculum/coursework/PAPER_TO_EXPERIMENT.md#a1--first-pass-figure-brief)):**
1. Two source-grounded transformation diagrams (`PP01` workflow + `PP04` component + proposed link);
2. The completed 8-row forensic comparison table;
3. Two 100-word figure briefs (one per paper);
4. Your prioritized 150–200 word response to the Black-Box AI Alignment scenario and three technical questions for the Processing strand.


## Return after the fundamentals

When you enter the Processing strand and Capstone Project `P01`, you will turn every question from this seminar into a concrete mathematical check:
- Study [PR01 · Images, coordinates, and metadata](../../notebooks/01_processing/01_images_coordinates_metadata.ipynb) (voxel lattices, homogeneous $4 \times 4$ affines, `RAS+` orientation, and physical mm units);
- Study [PR02 · Registration and resampling](../../notebooks/01_processing/02_registration_resampling.ipynb) and [PR03 · Nonlinear warps](../../notebooks/01_processing/03_nonlinear_warps.ipynb) (forward vs. pull-back coordinate mappings, cost functions, Jacobian determinants $|J_\phi|$, and diffeomorphic folding);
- Study [PR09 · Slice timing](../../notebooks/01_processing/09_slice_timing.ipynb), [PR10 · Motion](../../notebooks/01_processing/10_motion.ipynb), and [PR11 · Susceptibility distortion](../../notebooks/01_processing/11_susceptibility_distortion.ipynb) (the core 4D functional MRI corrections orchestrated by `fMRIPrep`);
- Study [PR15 · QC and censoring](../../notebooks/01_processing/15_qc_censoring.ipynb) and [PR21 · Workflow provenance and reports](../../notebooks/01_processing/21_workflow_provenance_reports.ipynb) (how Nipype DAGs, BIDS-Derivatives, and visual reports prevent silent pipeline corruption);
- Complete Capstone [P01 · Anatomical template workflow](../../notebooks/05_projects/01_anatomical_workflow.ipynb) (where you will execute grid resampling, physical-space Gaussian smoothing, and the exact linear-vs-nearest-neighbor mask interpolation failure on the real MNI152 template!).

When you return to your `R01` diagram after `P01`, annotate each arrow with the exact matrix shapes $(X, Y, Z)$, affine zoom factors $(\Delta x, \Delta y, \Delta z)$, and numerical invariants (`np.unique(mask) <= {0, 1}`, $|\det(\mathbf{A})|$) you verified. Keep your local experiment labeled honestly as a **study of transformation mechanics**, distinct from a full computational reproduction of the 100,000-scan BrainMorph training run.


## Tutor prompt and exit discussion

> **Supervisory Tutor Prompt for `R01`:**
> "Read only the paper excerpts and my transformation diagrams below. Before defining registration mathematically, ask me what changes when an image is aligned: its underlying anatomy, its coordinate mapping, or its resampled voxel grid. Check my diagram to verify which boxes represent (a) raw measurements, (b) estimated coordinate transforms, (c) interpolated image grids, or (d) human QC evidence. Do not guess missing metadata. Highlight one silent failure mode in my scenario audit (such as label interpolation or transform direction), and ask me how I would detect it visually and numerically."

### Seminar Exit Discussion Questions
1. Why can a software pipeline exit with status `0` ("finished without error") and still produce an anatomically or statistically unusable image?
2. What specific evidence separates a *visually plausible brain outline overlay* from the exact anatomical correspondence required by a cortical thickness (`PP07`) or voxelwise fMRI (`PP01`) hypothesis?
3. Why does `fMRIPrep` describe itself as **analysis-agnostic** (stopping before spatial smoothing or nuisance regression removal), and how does that boundary protect against the analytical variability you will study in `R03` (`PD01`)?

<details><summary>Instructor notes</summary>

### Grading & Supervision Guide for `R01`

1. **Workflow (`PP01`) vs. Component (`PP04`) Distinction:**
   - Reward learners who clearly see that `fMRIPrep` is a comprehensive BIDS orchestration pipeline covering both structural and 4D functional MRI (including slice timing, motion, $B_0$ susceptibility unwarping, confound extraction, and one-shot resampling), whereas `BrainMorph` is a specialized 3D registration module that estimates spatial transforms (Affine/TPS) via learned keypoints.
   - If a student proposes plugging `BrainMorph` into `fMRIPrep` (e.g., as a fast initialization or replacement for `antsRegistration` in the $T_1\text{w} \to \text{MNI}$ step), verify that they label this as **their own architectural proposal** and note what would still need to be tested (e.g., high-DOF cortical sulcal accuracy and impact on surface reconstruction).

2. **Priority Order in the "Black-Box AI Alignment" Scenario:**
   - **Priority 1 (Identity & Provenance):** Which exact source scan, participant ID, and target MNI template space (`ICBM152 2009c asym` vs. `MNI152NLin6Asym`, resolution in mm) were used? Without identity and reference-space provenance, no coordinate has meaning.
   - **Priority 2 (Transform Direction, DOF, & Affine Orientation):** Is the transform mapping Subject $\to$ MNI or MNI $\to$ Subject? Are axis codes preserved (`RAS+`) without an accidental Left-Right flip ($\det(\mathbf{A}) > 0$)? Was the transform rigid (volume-preserving) or affine/nonlinear (which scales native lesion volumes unless Jacobian-modulated)?
   - **Priority 3 (Interpolation Contract):** Continuous intensities require trilinear/Lanczos/spline interpolation, whereas discrete binary or multi-class masks (`{0, 1}`) **must** use nearest-neighbor interpolation (or thresholded probability maps) to prevent fractional boundary corruption and phantom integer labels (`P01`).
   - **Priority 4 (Visual & Numerical QC):** Orthogonal slice overlays of internal structures (ventricles, corpus callosum, pial/white borders—not just the outer skull silhouette!) plus quantitative invariants (brain mask volume before/after, unique label set check).

3. **Key Misconception to Correct Early:**
   - Students often assume "automated HTML report = guaranteed quality." Emphasize that `fMRIPrep` generates visual reports precisely *because* automated algorithms fail on motion artifacts, coil dropouts, and atypical anatomy—making human inspection of the report a mandatory scientific gate.

</details>

Next: [R02 prediction and generalization](02_why_generalization.ipynb).


<!-- agentic-guide -->
### Agentic Deliverable Supervision (R01 · macOS Goose + Ollama: Qwen & Gemma)

**Deliverable focus:** Paired input -> operation -> output -> evidence diagrams comparing fMRIPrep (`PP01`) workflow orchestration against BrainMorph (`PP04`) keypoint registration.

#### 1. Suggestive Prompt (in the spirit of what to ask — adapt, do not copy verbatim)
> Ask Goose/Gemma in the spirit of: *"Using only my notes on fMRIPrep (`PP01` Figure 1 & Figure 3) and BrainMorph (`PP04` Figure 1 & Figure 3), help me structure a side-by-side comparison table of their inputs, transformations, outputs, and evaluation metrics. Point out where a registration failure would hide if I only looked at a mean Dice score instead of a visual slice overlay."*

#### 2. What to Look For & How to Trace the Error Back Down
- **Immediate Red Flag:** The AI conflates fMRIPrep (a metadata-driven preprocessing orchestrator that calls classical registration/distortion tools and emits visual QC reports) with BrainMorph (a deep keypoint network that predicts 3D anatomical landmarks to drive affine/TPS/diffeomorphic warps), or claims that a high average Dice score guarantees topological validity.
- **Traceback & Fix:** Trace each claim back to `PP01` Fig. 1 vs. `PP04` Fig. 1–2 and separate *workflow orchestration* from *learned correspondence estimation*.

#### 3. Expected Outcome Range & Why It Must Look That Way
**Expected Range:** Two distinct 4-box transformation diagrams (`Input -> Operation -> Output -> Evidence`) plus $3$ targeted questions on coordinate spaces (`RAS+`), interpolation blurring, and Jacobian folding. **Why:** A preprocessing report (`PP01`) makes intermediate failure modes visible to a human auditor, whereas a learned registration model (`PP04`) must be evaluated on held-out anatomical correspondence and local deformation regularity ($\det(J) > 0$).
<!-- /agentic-guide -->
